# MIMIC-IV Exploratory Data Analysis (EDA)
## Khóa luận: Đánh giá mức độ sinh tồn của bệnh nhân tại ICU bằng học sâu

**Mục tiêu notebook:**
1. Thống kê mô tả cohort bệnh nhân (Table One)
2. Phân tích tỷ lệ missing data
3. Phân bố các chỉ số sinh hiệu
4. So sánh nhóm sống sót vs tử vong
5. Tương quan giữa các features
6. Phân tích chuỗi thời gian 48h

**Dữ liệu:** Parquet files từ MIMIC-IV (đã nén từ CSV)

In [ ]:
# ── CELL 1: Cài thư viện + Mount Drive ──────────────────────
# Bỏ comment nếu chạy trên Google Colab
# from google.colab import drive
# drive.mount('/content/drive')
# !pip install tableone pyarrow seaborn -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import OrderedDict
import gc
import warnings
warnings.filterwarnings('ignore')

# Cài đặt style biểu đồ chuyên nghiệp
plt.rcParams.update({
    'figure.figsize': (12, 6),
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})
sns.set_style("whitegrid")
print("✓ Libraries loaded")

## 1. Cấu hình đường dẫn

In [ ]:
# ── CELL 2: Đường dẫn dữ liệu ───────────────────────────────
# === CHỈNH Ở ĐÂY ===
DATA_DIR = r"E:\KLTN\mimiciv\3.1\parquet"  # Windows local
# DATA_DIR = "/content/drive/MyDrive/mimic4/parquet"  # Google Colab

import os
def load(folder, name):
    path = os.path.join(DATA_DIR, folder, name)
    if os.path.exists(path):
        df = pd.read_parquet(path)
        print(f"  ✓ {folder}/{name}: {df.shape[0]:,} rows × {df.shape[1]} cols")
        return df
    else:
        print(f"  ✗ {folder}/{name}: KHÔNG TÌM THẤY")
        return pd.DataFrame()

## 2. Load dữ liệu cơ bản

In [ ]:
# ── CELL 3: Load các bảng chính ───────────────────────────────
print("Loading core tables...")
patients   = load("hosp", "patients.parquet")
admissions = load("hosp", "admissions.parquet")
icustays   = load("icu",  "icustays.parquet")

# Merge thành cohort
cohort = (
    icustays
    .merge(admissions[["subject_id", "hadm_id", "admittime", "dischtime",
                        "hospital_expire_flag", "insurance", "race"]],
           on=["subject_id", "hadm_id"], how="inner")
    .merge(patients[["subject_id", "gender", "anchor_age", "dod"]],
           on="subject_id", how="inner")
)
# Chỉ người lớn, first ICU stay per admission
cohort = cohort[cohort["anchor_age"] >= 18]
cohort = (cohort.sort_values(["subject_id", "hadm_id", "intime"])
                .groupby(["subject_id", "hadm_id"], as_index=False).first())

# Tính các biến dẫn xuất
cohort["icu_los_hours"] = (
    pd.to_datetime(cohort["outtime"]) - pd.to_datetime(cohort["intime"])
).dt.total_seconds() / 3600
cohort["hosp_los_days"] = (
    pd.to_datetime(cohort["dischtime"]) - pd.to_datetime(cohort["admittime"])
).dt.total_seconds() / 86400
cohort["mortality"] = cohort["hospital_expire_flag"].astype(int)
cohort["age_group"] = pd.cut(cohort["anchor_age"],
                              bins=[17, 30, 50, 65, 80, 120],
                              labels=["18-30", "31-50", "51-65", "66-80", "80+"])
print(f"\nCohort: {len(cohort):,} ICU stays | "
      f"{cohort['subject_id'].nunique():,} unique patients")
print(f"Mortality: {cohort['mortality'].sum():,} ({cohort['mortality'].mean()*100:.1f}%)")

## 3. Table One — Bảng thống kê mô tả cohort

Bảng này là tiêu chuẩn bắt buộc trong mọi bài báo y khoa. So sánh đặc điểm nhân khẩu học giữa nhóm sống sót và tử vong.

In [ ]:
# ── CELL 4: Table One ─────────────────────────────────────────
try:
    from tableone import TableOne

    cohort_table = cohort.copy()
    cohort_table["Gender"] = cohort_table["gender"]
    cohort_table["Age"]    = cohort_table["anchor_age"]
    cohort_table["ICU LOS (hours)"]  = cohort_table["icu_los_hours"].round(1)
    cohort_table["Hospital LOS (days)"] = cohort_table["hosp_los_days"].round(1)
    cohort_table["Mortality"] = cohort_table["mortality"].map({0: "Survived", 1: "Died"})

    columns = ["Age", "Gender", "ICU LOS (hours)", "Hospital LOS (days)"]
    categorical = ["Gender"]
    groupby = "Mortality"

    table1 = TableOne(cohort_table, columns=columns,
                      categorical=categorical, groupby=groupby,
                      pval=True, missing=True)
    print(table1.tabulate(tablefmt="grid"))

except ImportError:
    print("tableone chưa cài. Chạy: pip install tableone")
    print("\n=== Thống kê thay thế ===")
    for col in ["anchor_age", "icu_los_hours", "hosp_los_days"]:
        for grp, label in [(0, "Survived"), (1, "Died")]:
            sub = cohort[cohort["mortality"] == grp][col]
            print(f"  {label} — {col}: mean={sub.mean():.1f}, std={sub.std():.1f}, "
                  f"median={sub.median():.1f}")

## 4. Phân bố nhân khẩu học

In [ ]:
# ── CELL 5: Biểu đồ phân bố tuổi theo nhóm sống/chết ─────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 4a. Tuổi
for label, grp in cohort.groupby("mortality"):
    name = "Tử vong" if label == 1 else "Sống sót"
    axes[0].hist(grp["anchor_age"], bins=30, alpha=0.6, label=name, density=True)
axes[0].set_xlabel("Tuổi")
axes[0].set_ylabel("Mật độ")
axes[0].set_title("Phân bố tuổi theo kết cục")
axes[0].legend()

# 4b. Giới tính
gender_mort = cohort.groupby(["gender", "mortality"]).size().unstack(fill_value=0)
gender_mort.plot(kind="bar", ax=axes[1], color=["#2196F3", "#f44336"])
axes[1].set_title("Giới tính × Kết cục")
axes[1].set_xlabel("Giới tính")
axes[1].set_ylabel("Số ca")
axes[1].legend(["Sống sót", "Tử vong"])
axes[1].tick_params(axis='x', rotation=0)

# 4c. Tỷ lệ tử vong theo nhóm tuổi
mort_by_age = cohort.groupby("age_group")["mortality"].mean() * 100
mort_by_age.plot(kind="bar", ax=axes[2], color="#f44336", edgecolor="black")
axes[2].set_title("Tỷ lệ tử vong theo nhóm tuổi (%)")
axes[2].set_xlabel("Nhóm tuổi")
axes[2].set_ylabel("Tỷ lệ tử vong (%)")
axes[2].tick_params(axis='x', rotation=0)
for i, v in enumerate(mort_by_age):
    axes[2].text(i, v + 0.5, f"{v:.1f}%", ha='center', fontsize=9)

plt.tight_layout()
img_dir = "img"
os.makedirs(img_dir, exist_ok=True)
plt.savefig(os.path.join(img_dir, "eda_demographics.png"), dpi=150, bbox_inches="tight")
plt.show()
print("✓ Saved: eda_demographics.png")

## 5. Phân bố thời gian nằm ICU

In [ ]:
# ── CELL 6: LOS distribution ──────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ICU LOS
for label, grp in cohort.groupby("mortality"):
    name = "Tử vong" if label == 1 else "Sống sót"
    los = grp["icu_los_hours"].clip(0, 500)
    axes[0].hist(los, bins=50, alpha=0.6, label=name, density=True)
axes[0].set_xlabel("ICU LOS (giờ)")
axes[0].set_ylabel("Mật độ")
axes[0].set_title("Phân bố thời gian nằm ICU")
axes[0].legend()
axes[0].axvline(x=48, color='red', linestyle='--', alpha=0.7, label='48h window')

# Hospital LOS
for label, grp in cohort.groupby("mortality"):
    name = "Tử vong" if label == 1 else "Sống sót"
    los = grp["hosp_los_days"].clip(0, 60)
    axes[1].hist(los, bins=50, alpha=0.6, label=name, density=True)
axes[1].set_xlabel("Hospital LOS (ngày)")
axes[1].set_ylabel("Mật độ")
axes[1].set_title("Phân bố thời gian nằm viện")
axes[1].legend()

plt.tight_layout()
img_dir = "img"
os.makedirs(img_dir, exist_ok=True)
plt.savefig(os.path.join(img_dir, "eda_los.png"), dpi=150, bbox_inches="tight")
plt.show()

# Thống kê LOS
print("\nThống kê ICU LOS (giờ):")
print(cohort.groupby("mortality")["icu_los_hours"].describe().round(1))

## 6. Phân tích Missing Rate — chartevents

Đây là phân tích quan trọng nhất trong EDA y tế ICU.
Tỷ lệ missing cao ở một feature nghĩa là bác sĩ ít đo chỉ số đó → ảnh hưởng chất lượng model.

In [ ]:
# ── CELL 7: Missing rate analysis ─────────────────────────────
# Phân tích này dùng trực tiếp từ dữ liệu preprocessed (chart_wide.parquet)
# thay vì đọc chartevents.parquet gốc (~30GB, quá lớn)

import gc

PROC_DIR = r"E:\KLTN\mimic4_processed"
# PROC_DIR = "/content/drive/MyDrive/mimic4_processed"

chart_wide_path = os.path.join(PROC_DIR, "chart_wide.parquet")

if os.path.exists(chart_wide_path):
    print("Loading chart_wide.parquet từ preprocessing output...")
    cw = pd.read_parquet(chart_wide_path)
    
    # Tính missing rate per feature (% giờ không có đo lường)
    vitals = ["HR", "RR", "Temp_C", "SBP", "DBP", "MBP", "SpO2",
              "GCS_total", "WBC", "HCT", "PLT", "pH", "FiO2",
              "PaO2", "PaCO2", "HCO3", "Glucose", "Creatinine",
              "BUN", "AST", "ALT", "CKMB", "BNP", "DDimer",
              "Na", "K", "Cl", "Mg", "Cortisol",
              "Bili_total", "Bili_direct", "Height", "Weight"]
    available = [v for v in vitals if v in cw.columns]
    
    missing_rates = {}
    for col in available:
        missing_rates[col] = cw[col].isna().mean() * 100
    
    missing_df = pd.Series(missing_rates).sort_values(ascending=True)
    
    fig, ax = plt.subplots(figsize=(10, max(6, len(missing_df) * 0.3)))
    colors = ["#f44336" if v > 80 else "#FF9800" if v > 50 else "#2196F3" for v in missing_df.values]
    bars = ax.barh(missing_df.index, missing_df.values, color=colors, edgecolor="black", linewidth=0.5)
    ax.set_xlabel("Missing Rate (%)")
    ax.set_title("Tỷ lệ thiếu dữ liệu per hour-bucket — Chart Vitals")
    for bar, val in zip(bars, missing_df.values):
        ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                f"{val:.1f}%", va='center', fontsize=8)
    ax.axvline(x=50, color='red', linestyle='--', alpha=0.5, label='50% threshold')
    ax.legend()
    plt.tight_layout()
    
    img_dir = "img"
    os.makedirs(img_dir, exist_ok=True)
    plt.savefig(os.path.join(img_dir, "eda_missing_rate_chart.png"), dpi=150, bbox_inches="tight")
    plt.show()
    print(f"✓ Saved: {img_dir}/eda_missing_rate_chart.png")
    
    del cw; gc.collect()
else:
    print(f"Chưa có chart_wide.parquet tại {PROC_DIR}")
    print("Chạy preprocessing trước (2_run.bat), rồi quay lại cell này.")


## 7. Phân tích Missing Rate — labevents

In [ ]:
# ── CELL 8: Missing rate — labevents ──────────────────────────
PROC_DIR = r"E:\KLTN\mimic4_processed"

lab_wide_path = os.path.join(PROC_DIR, "lab_wide.parquet")

if os.path.exists(lab_wide_path):
    print("Loading lab_wide.parquet từ preprocessing output...")
    lw = pd.read_parquet(lab_wide_path)
    
    lab_cols = ["WBC_lab", "HCT_lab", "PLT_lab",
                "pH_lab", "PaO2_lab", "PaCO2_lab", "HCO3_lab",
                "Creatinine_lab", "BUN_lab", "Albumin_lab",
                "Lactate_lab", "CRP_lab", "Ca_lab",
                "Na_lab", "K_lab", "Cl_lab", "Mg_lab",
                "AST_lab", "ALT_lab", "PT_lab", "PTT_lab",
                "Bili_total_lab", "Bili_direct_lab", "Bili_indirect_lab",
                "CKMB_lab", "BNP_lab", "DDimer_lab",
                "TSH_lab", "FT3_lab", "FT4_lab"]
    available = [c for c in lab_cols if c in lw.columns]
    
    lab_missing = {}
    for col in available:
        lab_missing[col] = lw[col].isna().mean() * 100
    
    lab_miss_df = pd.Series(lab_missing).sort_values(ascending=True)
    
    fig, ax = plt.subplots(figsize=(10, max(7, len(lab_miss_df) * 0.3)))
    colors = ["#f44336" if v > 80 else "#FF9800" if v > 50 else "#4CAF50" for v in lab_miss_df.values]
    bars = ax.barh(lab_miss_df.index, lab_miss_df.values, color=colors, edgecolor="black", linewidth=0.5)
    ax.set_xlabel("Missing Rate (%)")
    ax.set_title("Tỷ lệ thiếu dữ liệu per hour-bucket — Lab Tests")
    for bar, val in zip(bars, lab_miss_df.values):
        ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                f"{val:.1f}%", va='center', fontsize=8)
    ax.axvline(x=50, color='red', linestyle='--', alpha=0.5, label='50% threshold')
    ax.legend()
    plt.tight_layout()
    
    img_dir = "img"
    os.makedirs(img_dir, exist_ok=True)
    plt.savefig(os.path.join(img_dir, "eda_missing_rate_lab.png"), dpi=150, bbox_inches="tight")
    plt.show()
    print(f"✓ Saved: {img_dir}/eda_missing_rate_lab.png")
    
    del lw; gc.collect()
else:
    print(f"Chưa có lab_wide.parquet tại {PROC_DIR}")
    print("Chạy preprocessing trước.")


## 8. Phân bố các chỉ số sinh hiệu (Vital Signs)

Nếu đã chạy preprocessing và có file `cohort.parquet` + `chart_wide.parquet`,
dùng cell này. Nếu không, bỏ qua.

In [ ]:
# ── CELL 9: Vital signs distribution ──────────────────────────
# Thử load từ output preprocessing (nếu đã chạy)
PROC_DIR = r"E:\KLTN\mimic4_processed"
# PROC_DIR = "/content/drive/MyDrive/mimic4_processed"

chart_wide_path = os.path.join(PROC_DIR, "chart_wide.parquet")
cohort_path     = os.path.join(PROC_DIR, "cohort.parquet")

if os.path.exists(chart_wide_path) and os.path.exists(cohort_path):
    print("Loading preprocessed data...")
    cw = pd.read_parquet(chart_wide_path)
    co = pd.read_parquet(cohort_path)

    # Merge outcome
    cw = cw.merge(co[["AdmissionID", "Outcome"]], on="AdmissionID", how="inner")

    vitals = ["HR", "RR", "Temp_C", "SBP", "DBP", "SpO2", "Glucose", "pH"]
    available = [v for v in vitals if v in cw.columns]

    n_cols = 4
    n_rows = (len(available) + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4 * n_rows))
    axes = axes.flatten()

    for i, vital in enumerate(available):
        data_surv = cw[cw["Outcome"] == 0][vital].dropna()
        data_died = cw[cw["Outcome"] == 1][vital].dropna()
        
        # Clip outliers for visualization
        lo, hi = data_surv.quantile(0.01), data_surv.quantile(0.99)
        
        axes[i].hist(data_surv.clip(lo, hi), bins=50, alpha=0.5,
                     label="Survived", density=True, color="#2196F3")
        axes[i].hist(data_died.clip(lo, hi), bins=50, alpha=0.5,
                     label="Died", density=True, color="#f44336")
        axes[i].set_title(vital)
        axes[i].legend(fontsize=8)
    
    # Ẩn ô thừa
    for j in range(len(available), len(axes)):
        axes[j].set_visible(False)
    
    plt.suptitle("Phân bố chỉ số sinh hiệu: Sống sót vs Tử vong", y=1.02, fontsize=14)
    plt.tight_layout()
    img_dir = "img"
    os.makedirs(img_dir, exist_ok=True)
    plt.savefig(os.path.join(img_dir, "eda_vitals_distribution.png"), dpi=150, bbox_inches="tight")
    plt.show()
    print("✓ Saved: eda_vitals_distribution.png")
    del cw
else:
    print(f"Chưa có dữ liệu preprocessed tại {PROC_DIR}")
    print("Bỏ qua cell này hoặc chỉnh PROC_DIR cho đúng.")

## 9. Heatmap tương quan giữa các chỉ số (correlation matrix)

In [ ]:
# ── CELL 10: Correlation heatmap ──────────────────────────────
if os.path.exists(chart_wide_path):
    cw = pd.read_parquet(chart_wide_path)
    
    corr_cols = ["HR", "RR", "Temp_C", "SBP", "DBP", "MBP", "SpO2",
                 "Glucose", "pH", "FiO2", "WBC", "HCT", "PLT",
                 "BUN", "Creatinine", "AST", "ALT"]
    available_corr = [c for c in corr_cols if c in cw.columns]
    
    corr_matrix = cw[available_corr].corr()
    
    fig, ax = plt.subplots(figsize=(12, 10))
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
    sns.heatmap(corr_matrix, mask=mask, annot=True, fmt=".2f",
                cmap="RdBu_r", center=0, vmin=-1, vmax=1,
                square=True, linewidths=0.5, ax=ax,
                cbar_kws={"shrink": 0.8})
    ax.set_title("Ma trận tương quan giữa các chỉ số sinh hiệu ICU", fontsize=14)
    plt.tight_layout()
    img_dir = "img"
    os.makedirs(img_dir, exist_ok=True)
    plt.savefig(os.path.join(img_dir, "eda_correlation_heatmap.png"), dpi=150, bbox_inches="tight")
    plt.show()
    print("✓ Saved: eda_correlation_heatmap.png")
    del cw
else:
    print("Cần chart_wide.parquet từ preprocessing. Bỏ qua.")

## 10. Thống kê tổng hợp cho báo cáo

Cell này in ra các con số quan trọng bạn cần ghi vào chương Mô tả dữ liệu.

In [ ]:
# ── CELL 11: Summary statistics ───────────────────────────────
print("=" * 60)
print("  THỐNG KÊ TỔNG HỢP CHO BÁO CÁO KHÓA LUẬN")
print("=" * 60)
print()
print(f"Tổng số bệnh nhân (unique):     {cohort['subject_id'].nunique():,}")
print(f"Tổng số lượt nhập ICU:           {len(cohort):,}")
print(f"Tuổi trung bình ± SD:            {cohort['anchor_age'].mean():.1f} ± {cohort['anchor_age'].std():.1f}")
print(f"Tuổi trung vị (IQR):             {cohort['anchor_age'].median():.0f} "
      f"({cohort['anchor_age'].quantile(0.25):.0f}–{cohort['anchor_age'].quantile(0.75):.0f})")
print()
print(f"Giới tính nam:                   {(cohort['gender']=='M').sum():,} ({(cohort['gender']=='M').mean()*100:.1f}%)")
print(f"Giới tính nữ:                   {(cohort['gender']=='F').sum():,} ({(cohort['gender']=='F').mean()*100:.1f}%)")
print()
print(f"ICU LOS trung bình (giờ):        {cohort['icu_los_hours'].mean():.1f} ± {cohort['icu_los_hours'].std():.1f}")
print(f"Hospital LOS trung bình (ngày):  {cohort['hosp_los_days'].mean():.1f} ± {cohort['hosp_los_days'].std():.1f}")
print()
print(f"Tử vong trong bệnh viện:         {cohort['mortality'].sum():,} ({cohort['mortality'].mean()*100:.1f}%)")
print(f"Sống sót:                        {(cohort['mortality']==0).sum():,} ({(1-cohort['mortality'].mean())*100:.1f}%)")
print(f"Tỷ lệ mất cân bằng:             1:{(1-cohort['mortality'].mean())/cohort['mortality'].mean():.1f}")
print()
print("Tỷ lệ tử vong theo nhóm tuổi:")
for age_grp, rate in cohort.groupby("age_group")["mortality"].mean().items():
    n = (cohort["age_group"] == age_grp).sum()
    print(f"  {age_grp}: {rate*100:.1f}% (n={n:,})")
print()
print("=" * 60)
print("Copy các số liệu trên vào chương 'Mô tả dữ liệu' của khóa luận")
print("=" * 60)

## 11. Diagnoses — Top ICD codes

In [ ]:
# ── CELL 12: Top ICD diagnoses ────────────────────────────────
diag = load("hosp", "diagnoses_icd.parquet")
if not diag.empty:
    valid_hadm = set(cohort["hadm_id"].unique())
    diag = diag[diag["hadm_id"].isin(valid_hadm)]
    
    # Load mô tả ICD nếu có
    d_icd = load("hosp", "d_icd_diagnoses.parquet")
    if not d_icd.empty:
        diag = diag.merge(d_icd[["icd_code", "long_title"]], on="icd_code", how="left")
        top = diag["long_title"].value_counts().head(20)
    else:
        top = diag["icd_code"].value_counts().head(20)
    
    fig, ax = plt.subplots(figsize=(10, 8))
    top.plot(kind="barh", ax=ax, color="#FF9800", edgecolor="black")
    ax.set_xlabel("Số lượt chẩn đoán")
    ax.set_title("Top 20 chẩn đoán ICD phổ biến nhất trong cohort ICU")
    ax.invert_yaxis()
    plt.tight_layout()
    img_dir = "img"
    os.makedirs(img_dir, exist_ok=True)
    plt.savefig(os.path.join(img_dir, "eda_top_icd.png"), dpi=150, bbox_inches="tight")
    plt.show()
    print("✓ Saved: eda_top_icd.png")
else:
    print("diagnoses_icd.parquet không tìm thấy.")

---
## Tổng kết: Các file hình ảnh đã xuất

| File | Nội dung | Dùng trong chương |
|------|---------|-------------------|
| `eda_demographics.png` | Phân bố tuổi, giới tính, tỷ lệ tử vong theo tuổi | Mô tả dữ liệu |
| `eda_los.png` | Phân bố thời gian nằm ICU/bệnh viện | Mô tả dữ liệu |
| `eda_missing_rate_chart.png` | Tỷ lệ missing chartevents | Tiền xử lý dữ liệu |
| `eda_missing_rate_lab.png` | Tỷ lệ missing labevents | Tiền xử lý dữ liệu |
| `eda_vitals_distribution.png` | Phân bố vital signs sống/chết | Phân tích dữ liệu |
| `eda_correlation_heatmap.png` | Tương quan giữa các features | Phân tích dữ liệu |
| `eda_top_icd.png` | Top 20 chẩn đoán ICD | Mô tả dữ liệu |